In [1]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import os


## Get Data

In [2]:
# read csv file with neon transaction data
#neon_data = pd.read_csv('./data/transaction-report-2026-07-08-15-46-23.csv')
neon_data = pd.read_csv('./data/transaction-report-2026-08-04-11-26-07.csv', parse_dates=['Date'])
neon_data

,Status,Dispute Status,Items Subtotal,Total Excluding Tax,Subtotal,Taxes,Total,FX Rate,Fee Amount,Net Proceeds,...,Account Display Name,Account ID,Order Number,Date,SKUs,Items,Property Display Name,Property ID,Environment Display Name,Environment ID
0,succeeded,NaN,2.99,2.99,2.99,0.0,2.99,1.0,0.15,2.84,...,TM-SSUTMKKTQJWMLJXQ,8F134E18B224BCAA,2FLR-WGN8-LRK2,2026-08-04 11:15:08.534000+00:00,Feature-TipJar (qty: 1),1x Tip Jar (Feature-TipJar),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
1,succeeded,NaN,24.99,24.99,24.99,1.5,26.49,1.0,1.25,23.74,...,TM-OQILKXUQTXSKVMRT,B94D2AFB8CF23086,NTSL-S7QJ-QJSK,2026-08-04 11:14:33.437000+00:00,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,1x Garden Bundle (TimedAlbum-DailyOffers-2025Q...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
2,succeeded,NaN,0.99,0.99,0.99,0.0,0.99,1.0,0.05,0.94,...,TM-XVTSQRLWJPWKXUNR,95CF2E71E398ABDF,Z296-8Q5J-G4TW,2026-08-04 11:09:11.730000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
3,succeeded,NaN,0.99,0.99,0.99,0.0,0.99,1.0,0.05,0.94,...,TM-SQJRWOXQWJSKORKV,D2962A1E8F6E918A,9QR4-877J-FZ8P,2026-08-04 11:06:04.723000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
4,succeeded,NaN,2.99,2.99,2.99,0.2,3.19,1.0,0.15,2.84,...,TM-VIOTLSTKRSPKSWIK,20EA27A92BA3B60D,WGPM-3447-JQJ8,2026-08-04 11:02:37.514000+00:00,OfferTrack-TimedAlbum-IncPack-12P_299Bundle (q...,1x Offer (OfferTrack-TimedAlbum-IncPack-12P_29...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91643,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,TM-OLWONIMTMKVOJMUQ,8C416D24B4056E36,NaN,2026-02-16 11:57:13.269000+00:00,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
91644,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,TM-OLWONIMTMKVOJMUQ,8C416D24B4056E36,NaN,2026-02-16 09:41:49.456000+00:00,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
91645,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,TM-OLWONIMTMKVOJMUQ,8C416D24B4056E36,NaN,2026-02-16 09:35:00.550000+00:00,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
91646,expired,NaN,4.99,4.99,4.99,NaN,NaN,NaN,NaN,NaN,...,TM-OLWONIMTMKVOJMUQ,8C416D24B4056E36,NaN,2026-02-16 09:29:33.457000+00:00,bank-gems-gems_2 (qty: 1),1x Bescheidene Anzahl Edelsteine (bank-gems-ge...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617


In [3]:
refresh_data = False

In [4]:
query_location = './sql/neonpay_rp_IAPSuccess.sql'
parameters = {
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 2.61 GB when run.
Estimated query cost: $0.02


In [5]:
omni_rp_data = pd.DataFrame()
if refresh_data:
    omni_rp_data = bqc.get(query='./sql/neonpay_rp_IAPSuccess.sql', is_path=True, query_parameters=parameters)
    omni_rp_data.to_pickle('./data/neonpay_rp_IAPSuccess.pkl')

omni_rp_data = pd.read_pickle('./data/neonpay_rp_IAPSuccess.pkl')

In [6]:
omni_rp_data

,event_ts,user_id,purchase_id,transaction_id,transaction_store,outcome,iap_price_usd,currency_code,currency_amount
0,2026-07-31 21:43:16.056180,61BDFEC151AE7FE8,86188b53-1d9c-46b5-a76e-1f4fd856745b,37f7b81c-f923-4934-8b2d-a6e5b4a198bb,NeonPay,completed,2.99,USD,2.99
1,2026-07-31 09:15:44.112151,7A7109F385E5D9BC,8f58d66f-9099-4449-8143-b87a118e014e,87185f26-1ff4-4126-8dfb-19e51a555947,NeonPay,completed,2.99,USD,2.99
2,2026-07-31 23:02:12.624139,8FB82F0D7D9C021E,7d09f85a-e84d-4f88-b1ca-73cdb062826f,a25b2d03-5882-4603-8ae1-0f9ed7cb99c8,NeonPay,completed,2.99,USD,2.99
3,2026-07-31 06:40:29.437512,138C646772B40D6,3a872f1d-0442-4578-836e-249477d4ec70,215e5258-8e1e-47cf-bdc4-7cf751cbb9ba,NeonPay,completed,2.99,USD,2.99
4,2026-07-31 23:43:37.297882,783CC0BF5DC18EE,d604e535-c53a-455a-b051-a76e1d77c132,9f76acc6-79d1-47c7-bae4-8cb973067384,NeonPay,completed,2.99,USD,2.99
...,...,...,...,...,...,...,...,...,...
77749,2026-07-28 16:27:56.619319,67BB14AD23051968,1d90f19e-3330-446e-9e8f-1362779702ab,24b38cb1-4db1-481f-8bee-05468d712f69,NeonPay,completed,19.99,USD,19.99
77750,2026-07-28 15:16:10.074841,8220CF1FD537A551,30e520bf-ed19-4e1b-ae45-c5c26c2581f6,4bd84569-163a-4411-88ae-32c28466953f,NeonPay,completed,19.99,USD,19.99
77751,2026-07-28 22:26:07.838073,EF41CAA7AEAE6DED,d9a211a5-8e1c-4805-93d2-08002fb226ee,18c6bab5-3117-4e98-9686-a2e2f69054ae,NeonPay,completed,49.99,USD,49.99
77752,2026-07-28 11:49:26.018705,8476166929B5332D,acd80c6d-f842-4ddf-b6b8-efa91d2c818f,3e99a897-6a5d-4045-a914-8ef758d13c95,NeonPay,completed,49.99,USD,49.99


In [7]:
query_location = './sql/neonpay_fact_IAPProduct.sql'
parameters = {
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 197.6 MB when run.
Estimated query cost: $0.00


In [8]:
omni_product_data = pd.DataFrame()
if refresh_data:
    omni_product_data = bqc.get(query='./sql/neonpay_fact_IAPProduct.sql', is_path=True, query_parameters=parameters)
    omni_product_data.to_pickle('./data/neonpay_fact_IAPProduct_data.pkl')

omni_product_data = pd.read_pickle('./data/neonpay_fact_IAPProduct_data.pkl')

In [9]:
omni_product_data

,user_id,transaction_store,product_category,product_type,product_theme,product_id,product_price_group,iap_price_usd,dt,usd_iap_revenue,usd_net_iap_revenue,usd_iap_price_revenue,usd_net_iap_price_revenue,n_trans,loading_timestamp
0,21B578A30B74D016,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,3.98,3.7810,3.98,3.7810,2,2026-05-19 02:23:52.357046+00:00
1,6825E202BFCAA26,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,1.99,1.8905,1.99,1.8905,1,2026-05-19 02:23:52.357046+00:00
2,961383D4464E62B9,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,1.99,1.8905,1.99,1.8905,1,2026-05-19 02:23:52.357046+00:00
3,B80ADA3ACCFEC958,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,1.99,1.8905,1.99,1.8905,1,2026-05-19 02:23:52.357046+00:00
4,D816BD6258CECDC5,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,1.99,1.8905,1.99,1.8905,1,2026-05-19 02:23:52.357046+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76141,6FFF3BD86DC26E98,NeonPay,gems,Bank,Gems,bank-gems-gems_4,gems_1099_1999,19.99,2026-04-24,19.99,18.9905,19.99,18.9905,1,2026-04-29 02:45:03.207483+00:00
76142,4F38777762EFE50,NeonPay,gems,Bank,Gems,bank-gems-gems_4,gems_1099_1999,19.99,2026-04-24,19.99,18.9905,19.99,18.9905,1,2026-04-29 02:45:03.207483+00:00
76143,D230F9C4E0E7EE23,NeonPay,gems,Bank,Gems,bank-gems-gems_4,gems_1099_1999,19.99,2026-04-24,19.99,18.9905,19.99,18.9905,1,2026-04-29 02:45:03.207483+00:00
76144,268C5D5ED7412BFC,NeonPay,gems,Bank,Gems,bank-gems-gems_4,gems_1099_1999,19.99,2026-04-24,19.99,18.9905,19.99,18.9905,1,2026-04-29 02:45:03.207483+00:00


In [10]:
query_location = './sql/neonpay_fact_PurchaseRates.sql'
parameters = {
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 32.7 MB when run.
Estimated query cost: $0.00


In [11]:
omni_purchase_data = pd.DataFrame()
if refresh_data:
    omni_purchase_data = bqc.get(query='./sql/neonpay_fact_PurchaseRates.sql', is_path=True, query_parameters=parameters)
    omni_purchase_data.to_pickle('./data/neonpay_fact_PurchaseRates_data.pkl')

omni_purchase_data = pd.read_pickle('./data/neonpay_fact_PurchaseRates_data.pkl')

In [12]:
omni_purchase_data

,user_id,dt,product_id,product_category,price,n_offered_dtc,purchased,purchased_dtc,n_trans_dtc,n_trans,usd_iap_price_revenue_dtc,usd_iap_price_revenue,usd_net_iap_price_revenue_dtc,usd_net_iap_price_revenue,loading_timestamp
0,54706789BBFF73F6,2026-06-26,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,0.99,1,1,1,1,1,0.99,0.99,0.9405,0.9405,2026-07-01 02:30:52.458144+00:00
1,5001A0EEE2F18A,2026-06-26,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,0.99,1,1,1,1,1,0.99,0.99,0.9405,0.9405,2026-07-01 02:30:52.458144+00:00
2,B11481763188CDCB,2026-06-26,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-07-01 02:30:52.458144+00:00
3,1C92F681C364A087,2026-06-26,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-07-01 02:30:52.458144+00:00
4,5E9D4B0E728A681A,2026-06-26,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,2.99,1,1,1,1,1,2.99,2.99,2.8405,2.8405,2026-07-01 02:30:52.458144+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67937,1652313C768384E,2026-06-01,OfferTrack-TimedAlbum-IncPack-32P_1999Bundle,offer_track,19.99,1,1,1,1,1,19.99,19.99,18.9905,18.9905,2026-06-06 02:40:50.516719+00:00
67938,1652313C768384E,2026-06-01,OfferTrack-TimedAlbum-IncPack-36P_2499Bundle,offer_track,24.99,1,1,1,1,1,24.99,24.99,23.7405,23.7405,2026-06-06 02:40:50.516719+00:00
67939,7CC6CCD7E97F74FB,2026-06-01,OfferTrack-TimedAlbum-IncPack-36P_2499Bundle,offer_track,24.99,1,1,1,1,1,24.99,24.99,23.7405,23.7405,2026-06-06 02:40:50.516719+00:00
67940,1652313C768384E,2026-06-01,OfferTrack-TimedAlbum-IncPack-40P_2999Bundle,offer_track,29.99,1,1,1,1,1,29.99,29.99,28.4905,28.4905,2026-06-06 02:40:50.516719+00:00


## Check data

In [13]:
neon_data.groupby(['Currency']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,Currency,count
7,USD,91572
4,GBP,52
2,CNY,9
3,EUR,8
1,CAD,4
0,AUD,1
5,NOK,1
6,PEN,1


In [14]:
omni_rp_data.groupby(['currency_code']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,currency_code,count
7,USD,77579
3,GBP,120
5,JPY,31
1,CAD,7
2,EUR,7
6,NOK,7
4,INR,2
0,AUD,1


In [15]:
neon_data.groupby(['Status']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,Status,count
5,succeeded,77961
1,expired,11288
2,failed,2257
3,incomplete,73
0,disputed,45
4,refunded,24


In [16]:
omni_rp_data.groupby(['outcome']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,outcome,count
0,completed,77754


## Process data

In [17]:
start_date='2026-05-01'
end_date='2026-08-01'

In [23]:
neon_data

,Status,Dispute Status,Items Subtotal,Total Excluding Tax,Subtotal,Taxes,Total,FX Rate,Fee Amount,Net Proceeds,...,Order Number,Date,SKUs,Items,Property Display Name,Property ID,Environment Display Name,Environment ID,Date_trunc,Date_month
0,succeeded,NaN,2.99,2.99,2.99,0.0,2.99,1.0,0.15,2.84,...,2FLR-WGN8-LRK2,2026-08-04 11:15:08.534000+00:00,Feature-TipJar (qty: 1),1x Tip Jar (Feature-TipJar),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
1,succeeded,NaN,24.99,24.99,24.99,1.5,26.49,1.0,1.25,23.74,...,NTSL-S7QJ-QJSK,2026-08-04 11:14:33.437000+00:00,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,1x Garden Bundle (TimedAlbum-DailyOffers-2025Q...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
2,succeeded,NaN,0.99,0.99,0.99,0.0,0.99,1.0,0.05,0.94,...,Z296-8Q5J-G4TW,2026-08-04 11:09:11.730000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
3,succeeded,NaN,0.99,0.99,0.99,0.0,0.99,1.0,0.05,0.94,...,9QR4-877J-FZ8P,2026-08-04 11:06:04.723000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
4,succeeded,NaN,2.99,2.99,2.99,0.2,3.19,1.0,0.15,2.84,...,WGPM-3447-JQJ8,2026-08-04 11:02:37.514000+00:00,OfferTrack-TimedAlbum-IncPack-12P_299Bundle (q...,1x Offer (OfferTrack-TimedAlbum-IncPack-12P_29...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91643,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,NaN,2026-02-16 11:57:13.269000+00:00,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-02-16,2026-02
91644,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,NaN,2026-02-16 09:41:49.456000+00:00,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-02-16,2026-02
91645,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,NaN,2026-02-16 09:35:00.550000+00:00,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-02-16,2026-02
91646,expired,NaN,4.99,4.99,4.99,NaN,NaN,NaN,NaN,NaN,...,NaN,2026-02-16 09:29:33.457000+00:00,bank-gems-gems_2 (qty: 1),1x Bescheidene Anzahl Edelsteine (bank-gems-ge...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-02-16,2026-02


In [24]:
neon_data_fixed = pd.DataFrame()
neon_data_fixed = neon_data

neon_data_fixed['Date'] = pd.to_datetime(neon_data_fixed['Date'])
neon_data_fixed['Date_trunc'] = pd.to_datetime(neon_data_fixed['Date']).dt.date
neon_data_fixed['Date_month'] = pd.to_datetime(neon_data_fixed['Date']).dt.to_period('M')


neon_data_fixed = neon_data_fixed[neon_data_fixed.Status.isin(['succeeded'])]
neon_data_fixed = neon_data_fixed[(neon_data_fixed.Date_trunc >= pd.to_datetime(start_date).date()) & (neon_data_fixed.Date_trunc < pd.to_datetime(end_date).date())]
neon_data_fixed.sort_values(by='Date', ascending=True, inplace=True)
neon_data_fixed[['Date','Date_trunc','Date_month','Account ID','Items Subtotal','Total','Total Excluding Tax','Subtotal','Fee Amount','Net Proceeds','Currency']]

/var/folders/vj/nwmxdwwn0hl5rtj_xcywvvpc0000gn/T/ipykernel_60384/1839907189.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  neon_data_fixed['Date_month'] = pd.to_datetime(neon_data_fixed['Date']).dt.to_period('M')


,Date,Date_trunc,Date_month,Account ID,Items Subtotal,Total,Total Excluding Tax,Subtotal,Fee Amount,Net Proceeds,Currency
61185,2026-05-01 00:00:53.608000+00:00,2026-05-01,2026-05,DAE598C198F57109,6.99,6.99,6.99,6.99,0.35,6.64,USD
61184,2026-05-01 00:04:07.247000+00:00,2026-05-01,2026-05,E8E6CA67C07B945,6.99,6.99,6.99,6.99,0.35,6.64,USD
61183,2026-05-01 00:05:22.955000+00:00,2026-05-01,2026-05,EF449C1A73AD4ACB,19.99,21.66,19.99,19.99,1.00,18.99,USD
61181,2026-05-01 00:05:30.936000+00:00,2026-05-01,2026-05,48E2C02270F7B99A,19.99,21.63,19.99,19.99,1.00,18.99,USD
61180,2026-05-01 00:08:10.213000+00:00,2026-05-01,2026-05,8BE232F34550A513,24.99,24.99,24.99,24.99,1.25,23.74,USD
...,...,...,...,...,...,...,...,...,...,...,...
2218,2026-07-31 23:55:45.204000+00:00,2026-07-31,2026-07,4E5390ADB3A7D088,2.99,2.99,2.99,2.99,0.15,2.84,USD
2217,2026-07-31 23:58:07.530000+00:00,2026-07-31,2026-07,D05B67FBE921BCC9,4.99,4.99,4.99,4.99,0.25,4.74,USD
2216,2026-07-31 23:58:59.804000+00:00,2026-07-31,2026-07,35FA7AC14191C1FE,1.99,2.15,1.99,1.99,0.10,1.89,USD
2215,2026-07-31 23:59:11.094000+00:00,2026-07-31,2026-07,AEB098695CFD8369,2.99,2.99,2.99,2.99,0.15,2.84,USD


In [25]:
neon_data_fixed[neon_data_fixed.Currency!='USD']

,Status,Dispute Status,Items Subtotal,Total Excluding Tax,Subtotal,Taxes,Total,FX Rate,Fee Amount,Net Proceeds,...,Order Number,Date,SKUs,Items,Property Display Name,Property ID,Environment Display Name,Environment ID,Date_trunc,Date_month
60318,succeeded,NaN,9.09,7.57,7.57,1.52,9.09,1.357552,0.52,9.76,...,FRF4-36DT-BKHS,2026-05-02 18:08:27.899000+00:00,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,1x Garden Bundle (TimedAlbum-DailyOffers-2025Q...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-02,2026-05
59772,succeeded,NaN,11.89,9.91,9.91,1.98,11.89,1.357552,0.68,12.77,...,MLYS-DSMM-JC6C,2026-05-03 18:19:25.184000+00:00,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,1x Garden Bundle (TimedAlbum-DailyOffers-2025Q...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-03,2026-05
59331,succeeded,NaN,0.90,0.75,0.75,0.15,0.90,1.354244,0.06,0.96,...,M87K-DRQ7-RCKX,2026-05-04 11:27:22.543000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-04,2026-05
59282,succeeded,NaN,0.90,0.75,0.75,0.15,0.90,1.356114,0.06,0.96,...,BQ9K-BH5P-MH6W,2026-05-04 12:41:31.759000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-04,2026-05
59050,succeeded,NaN,9.09,7.57,7.57,1.52,9.09,1.353596,0.52,9.73,...,5RXR-3ZPY-QZ5B,2026-05-04 19:12:33.090000+00:00,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,1x Garden Bundle (TimedAlbum-DailyOffers-2025Q...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-04,2026-05
57897,succeeded,NaN,9.09,7.57,7.57,1.52,9.09,1.359000,0.52,9.77,...,K9V4-PHMN-BQ2W,2026-05-06 07:42:07.622000+00:00,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,1x Garden Bundle (TimedAlbum-DailyOffers-2025Q...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-06,2026-05
55708,succeeded,NaN,4.79,4.35,4.35,0.44,4.79,0.724690,0.16,2.99,...,92WB-VVV3-KVMN,2026-05-10 00:50:07.524000+00:00,OfferTrack-TimedAlbum-IncPack-12P_299Bundle (q...,1x Offer (OfferTrack-TimedAlbum-IncPack-12P_29...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-10,2026-05
51043,succeeded,NaN,45.39,37.82,37.82,7.57,45.39,1.332498,2.52,47.88,...,VDHW-695B-96VY,2026-05-16 21:46:45.974000+00:00,bank-gems-gems_5 (qty: 1),1x Huge amount of gems (bank-gems-gems_5),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-16,2026-05
50688,succeeded,NaN,0.90,0.75,0.75,0.15,0.90,1.332498,0.05,0.95,...,NG7B-8BNS-GGWX,2026-05-17 12:41:02.481000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-17,2026-05
50005,succeeded,NaN,0.90,0.75,0.75,0.15,0.90,1.338555,0.05,0.95,...,5JWR-6Z97-75CR,2026-05-18 12:11:48.765000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-05-18,2026-05


In [26]:
test_user_day = neon_data_fixed.groupby(['Account ID','Date_trunc','Items Subtotal']).size().reset_index(name='count').sort_values(by='count', ascending=False)
test_user_day[test_user_day['count']>1]

,Account ID,Date_trunc,Items Subtotal,count
4321,21B578A30B74D016,2026-07-27,1.99,6
16581,647FA14B8BCEDFFC,2026-06-15,2.99,5
45641,ED2E47BD0C5BB5FD,2026-05-01,1.99,5
18565,6C68FCA58F8E0308,2026-06-15,1.99,5
11018,46DD609EA9756006,2026-05-25,4.99,4
...,...,...,...,...
18747,6D7C19E64C94E49D,2026-05-16,1.99,2
18751,6D7C19E64C94E49D,2026-06-03,4.99,2
24349,8C2210D9D44053D2,2026-07-29,2.99,2
159,10A6B4965E403448,2026-06-10,1.99,2


In [27]:
omni_rp_fixed = pd.DataFrame()
omni_rp_fixed = omni_rp_data

omni_rp_fixed['dt_trunc'] = pd.to_datetime(omni_rp_fixed['event_ts']).dt.date
omni_rp_fixed['dt_month'] = pd.to_datetime(omni_rp_fixed['event_ts']).dt.to_period('M')
omni_rp_fixed = omni_rp_fixed[omni_rp_fixed.transaction_store=='NeonPay']
omni_rp_fixed = omni_rp_fixed[(omni_rp_fixed.dt_trunc >= pd.to_datetime(start_date).date()) & (omni_rp_fixed.dt_trunc < pd.to_datetime(end_date).date())]
omni_rp_fixed.sort_values(by='event_ts', ascending=True, inplace=True)
omni_rp_fixed

,event_ts,user_id,purchase_id,transaction_id,transaction_store,outcome,iap_price_usd,currency_code,currency_amount,dt_trunc,dt_month
49834,2026-05-01 00:01:29.016066,DAE598C198F57109,74d66490-8db9-479e-802b-0dbb897fd17a,e92ce86b-6eda-44ae-a705-6e4ecfa798fe,NeonPay,completed,6.99,USD,6.99,2026-05-01,2026-05
49840,2026-05-01 00:05:00.841838,E8E6CA67C07B945,95133986-dcf3-45d5-ab79-9f79bb342947,33d874a4-87bb-45d3-a3c4-8ba1d0b9d5dc,NeonPay,completed,6.99,USD,6.99,2026-05-01,2026-05
50053,2026-05-01 00:06:29.704513,48E2C02270F7B99A,2307e71d-e0d3-4bb6-ac8d-ccfb0314ea62,4ae011f1-8f6b-41b6-b7cc-d3daec386a04,NeonPay,completed,19.99,USD,19.99,2026-05-01,2026-05
50042,2026-05-01 00:06:51.732503,EF449C1A73AD4ACB,d811e8e1-9e42-47ec-a359-e966102f7069,557a85bc-c8cb-44b3-b4c0-5fafbe9f7d73,NeonPay,completed,19.99,USD,19.99,2026-05-01,2026-05
49967,2026-05-01 00:09:45.610088,8BE232F34550A513,f11eece0-a4a6-42b2-b52a-96bb054ac7d2,ebb84433-aeff-4e15-b493-597574fafd29,NeonPay,completed,24.99,USD,24.99,2026-05-01,2026-05
...,...,...,...,...,...,...,...,...,...,...,...
93,2026-07-31 23:56:17.308325,528DF3DD864F878B,40b2f90c-7f05-4ba3-bc81-f59fac54cec4,2fa41020-4a72-4618-ab2c-b54512c5faf4,NeonPay,completed,2.99,USD,2.99,2026-07-31,2026-07
306,2026-07-31 23:56:22.596599,4E5390ADB3A7D088,f3096a55-2a63-46f8-bc78-d22caccfb703,28f2a86b-ca48-4e6b-bf04-0d1f40617907,NeonPay,completed,2.99,USD,2.99,2026-07-31,2026-07
169,2026-07-31 23:58:38.976111,D05B67FBE921BCC9,47ab720b-92c3-4fd2-b23b-8ba870d2ca29,9c3b7b6c-4264-4e73-8110-a9ded7237c16,NeonPay,completed,4.99,USD,4.99,2026-07-31,2026-07
41,2026-07-31 23:59:34.108668,AEB098695CFD8369,9b121b9b-9da8-496e-a1f6-9517dd3dac8f,27c4d363-bcad-43d2-b058-1e13e8ee4d8d,NeonPay,completed,2.99,USD,2.99,2026-07-31,2026-07


In [28]:
omni_product_fixed = pd.DataFrame()
omni_product_fixed = omni_product_data

omni_product_fixed['dt_month'] = pd.to_datetime(omni_product_fixed['dt']).dt.to_period('M')
#omni_product_fixed = omni_product_fixed[omni_product_fixed.transaction_store=='NeonPay']
omni_product_fixed = omni_product_fixed[(omni_product_fixed.dt >= pd.to_datetime(start_date).date()) & (omni_product_fixed.dt < pd.to_datetime(end_date).date())]
omni_product_fixed.sort_values(by='dt', ascending=True, inplace=True)
omni_product_fixed

/var/folders/vj/nwmxdwwn0hl5rtj_xcywvvpc0000gn/T/ipykernel_60384/3350329144.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_product_fixed.sort_values(by='dt', ascending=True, inplace=True)


,user_id,transaction_store,product_category,product_type,product_theme,product_id,product_price_group,iap_price_usd,dt,usd_iap_revenue,usd_net_iap_revenue,usd_iap_price_revenue,usd_net_iap_price_revenue,n_trans,loading_timestamp,dt_month
65922,5F99BFC6BCB9492F,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_0099_0499,1.99,2026-05-01,1.99,1.8905,1.99,1.8905,1,2026-05-06 01:59:20.224165+00:00,2026-05
66064,84F4FD7EA8144154,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_1099_1999,14.99,2026-05-01,14.99,14.2405,14.99,14.2405,1,2026-05-06 01:59:20.224165+00:00,2026-05
66065,91E02417BFA42438,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_1099_1999,14.99,2026-05-01,14.99,14.2405,14.99,14.2405,1,2026-05-06 01:59:20.224165+00:00,2026-05
66066,76C65ACF385CCCCA,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_1099_1999,14.99,2026-05-01,14.99,14.2405,14.99,14.2405,1,2026-05-06 01:59:20.224165+00:00,2026-05
66067,1203B4ECAB631538,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_1099_1999,14.99,2026-05-01,14.99,14.2405,14.99,14.2405,1,2026-05-06 01:59:20.224165+00:00,2026-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10372,5A9688ABBC38DD2B,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_0599_0999,6.99,2026-07-31,6.99,6.6405,6.99,6.6405,1,2026-08-04 02:38:16.100826+00:00,2026-07
10373,E17FA947037C0CD,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_0599_0999,6.99,2026-07-31,6.99,6.6405,6.99,6.6405,1,2026-08-04 02:38:16.100826+00:00,2026-07
10374,F08091465F4CA39D,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_0599_0999,6.99,2026-07-31,6.99,6.6405,6.99,6.6405,1,2026-08-04 02:38:16.100826+00:00,2026-07
10360,7EF5C36661176DC0,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_0599_0999,5.99,2026-07-31,5.99,5.6905,5.99,5.6905,1,2026-08-04 02:38:16.100826+00:00,2026-07


In [29]:
omni_purchase_fixed = pd.DataFrame()
omni_purchase_fixed = omni_purchase_data

omni_purchase_fixed['dt_month'] = pd.to_datetime(omni_purchase_fixed['dt']).dt.to_period('M')
omni_purchase_fixed = omni_purchase_fixed[(omni_purchase_fixed.dt >= pd.to_datetime(start_date).date()) & (omni_purchase_fixed.dt < pd.to_datetime(end_date).date())]
omni_purchase_fixed.sort_values(by='dt', ascending=True, inplace=True)
omni_purchase_fixed

/var/folders/vj/nwmxdwwn0hl5rtj_xcywvvpc0000gn/T/ipykernel_60384/2093879705.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_purchase_fixed.sort_values(by='dt', ascending=True, inplace=True)


,user_id,dt,product_id,product_category,price,n_offered_dtc,purchased,purchased_dtc,n_trans_dtc,n_trans,usd_iap_price_revenue_dtc,usd_iap_price_revenue,usd_net_iap_price_revenue_dtc,usd_net_iap_price_revenue,loading_timestamp,dt_month
23651,95CF2E71E398ABDF,2026-05-01,OfferTrack-TimedAlbum-IncPack-4P_099Bundle,offer_track,0.99,1,1,1,1,1,0.99,0.99,0.9405,0.9405,2026-05-06 02:15:42.808482+00:00,2026-05
23818,C4E560588EFDCC27,2026-05-01,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,12.99,2,1,1,1,1,12.99,12.99,12.3405,12.3405,2026-05-06 02:15:42.808482+00:00,2026-05
23817,EE9B3A8ED5D13A2B,2026-05-01,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,12.99,1,1,1,1,1,12.99,12.99,12.3405,12.3405,2026-05-06 02:15:42.808482+00:00,2026-05
23816,CF1E61E8D5B3E04D,2026-05-01,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,12.99,1,1,1,1,1,12.99,12.99,12.3405,12.3405,2026-05-06 02:15:42.808482+00:00,2026-05
23815,7DE50A5DFD9CBF13,2026-05-01,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,12.99,1,1,1,1,1,12.99,12.99,12.3405,12.3405,2026-05-06 02:15:42.808482+00:00,2026-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6779,7BFC91AE148B3F10,2026-07-31,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,1.99,2,1,1,1,2,1.99,3.98,1.8905,3.2835,2026-08-04 03:02:36.756741+00:00,2026-07
6778,9CD16D137054409F,2026-07-31,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-08-04 03:02:36.756741+00:00,2026-07
6777,4F21EF45B44C02DE,2026-07-31,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-08-04 03:02:36.756741+00:00,2026-07
6775,A7AB347F5910D27C,2026-07-31,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,0.99,1,1,1,1,1,0.99,0.99,0.9405,0.9405,2026-08-04 03:02:36.756741+00:00,2026-07


## Reconcile data

In [30]:
neon_data

,Status,Dispute Status,Items Subtotal,Total Excluding Tax,Subtotal,Taxes,Total,FX Rate,Fee Amount,Net Proceeds,...,Order Number,Date,SKUs,Items,Property Display Name,Property ID,Environment Display Name,Environment ID,Date_trunc,Date_month
0,succeeded,NaN,2.99,2.99,2.99,0.0,2.99,1.0,0.15,2.84,...,2FLR-WGN8-LRK2,2026-08-04 11:15:08.534000+00:00,Feature-TipJar (qty: 1),1x Tip Jar (Feature-TipJar),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
1,succeeded,NaN,24.99,24.99,24.99,1.5,26.49,1.0,1.25,23.74,...,NTSL-S7QJ-QJSK,2026-08-04 11:14:33.437000+00:00,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,1x Garden Bundle (TimedAlbum-DailyOffers-2025Q...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
2,succeeded,NaN,0.99,0.99,0.99,0.0,0.99,1.0,0.05,0.94,...,Z296-8Q5J-G4TW,2026-08-04 11:09:11.730000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
3,succeeded,NaN,0.99,0.99,0.99,0.0,0.99,1.0,0.05,0.94,...,9QR4-877J-FZ8P,2026-08-04 11:06:04.723000+00:00,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
4,succeeded,NaN,2.99,2.99,2.99,0.2,3.19,1.0,0.15,2.84,...,WGPM-3447-JQJ8,2026-08-04 11:02:37.514000+00:00,OfferTrack-TimedAlbum-IncPack-12P_299Bundle (q...,1x Offer (OfferTrack-TimedAlbum-IncPack-12P_29...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-08-04,2026-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91643,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,NaN,2026-02-16 11:57:13.269000+00:00,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-02-16,2026-02
91644,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,NaN,2026-02-16 09:41:49.456000+00:00,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-02-16,2026-02
91645,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,NaN,2026-02-16 09:35:00.550000+00:00,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-02-16,2026-02
91646,expired,NaN,4.99,4.99,4.99,NaN,NaN,NaN,NaN,NaN,...,NaN,2026-02-16 09:29:33.457000+00:00,bank-gems-gems_2 (qty: 1),1x Bescheidene Anzahl Edelsteine (bank-gems-ge...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617,2026-02-16,2026-02


In [33]:
neon_data_recon = neon_data_fixed
neon_data_recon = neon_data_recon.groupby(['Date_month']).agg(
    iap_revenue = ('Total', 'sum'),
    iap_revenue_excl_tax = ('Items Subtotal', 'sum'),
    iap_net_revenue = ('Net Proceeds', 'sum'),
    
)

neon_data_recon

,iap_revenue,iap_revenue_excl_tax,iap_net_revenue
Date_month,,,
2026-05,149859.78,145609.81,138334.26
2026-06,134562.51,130486.97,123955.44
2026-07,142831.07,138482.77,131562.71


In [ ]:
omni_rp_recon = omni_rp_fixed
omni_rp_recon = omni_rp_recon.groupby(['dt_month']).agg(
    iap_revenue = ('iap_price_usd', 'sum')
)

omni_rp_recon['iap_net_revenue'] = omni_rp_recon['iap_revenue'] * 0.95

omni_rp_recon

In [ ]:
omni_product_recon = omni_product_fixed
omni_product_recon = omni_product_recon.groupby(['dt_month']).agg(
    iap_revenue = ('usd_iap_price_revenue', 'sum'),
    iap_net_revenue = ('usd_net_iap_revenue', 'sum')
)

omni_product_recon

In [ ]:
omni_purchase_recon = omni_purchase_fixed
omni_purchase_recon = omni_purchase_recon.groupby(['dt_month']).agg(
    iap_revenue = ('usd_iap_price_revenue_dtc', 'sum'),
    iap_net_revenue = ('usd_net_iap_price_revenue_dtc', 'sum')
)

omni_purchase_recon